# 06 KIỂM THỬ BỘ CHUYỂN TRẠNG THÁI ENGINE (ENGINE TESTING)

Notebook này thực hiện kiểm thử toàn diện module `commander_game_engine.engine`:
1. **Tính bất biến của trạng thái (State Immutability)**: Đảm bảo không sửa đổi trạng thái truyền vào.
2. **Thực thi nước đi thường (`MOVE`)**: Cập nhật tọa độ, giải phóng ô cũ, chuyển lượt thi đấu.
3. **Thực thi nước ăn quân (`CAPTURE`)**: Tiêu diệt quân địch, thế chỗ, tính điểm chuẩn theo `PIECE_POINTS`.
4. **Thực thi bắn hỏa lực tại chỗ (`BOMBARD`)**: Diệt mục tiêu tầm xa, giữ nguyên vị trí, cộng điểm.
5. **Điều kiện kết thúc ván đấu (Terminal Victory)**: Phát hiện Tư lệnh bị tiêu diệt, công nhận người thắng, phát sinh sự kiện `GAME_WON`.
6. **Từ chối hành động không hợp lệ**: Giữ nguyên vẹn trạng thái khi gặp hành động sai luật.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from copy import deepcopy

# Cấu hình nạp module
cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / "packages").is_dir() and (p / "research").is_dir()), None)
if str(PROJECT_ROOT / "packages") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "packages"))

from commander_game_engine.board import Board, Position, Terrain
from commander_game_engine.game_state import GameState, Piece, PieceType, Player
from commander_game_engine.initial_setup import create_initial_state
from commander_game_engine.actions import ActionType, AtomicAction
from commander_game_engine.engine import (
    apply_action,
    EngineResult,
    EngineEvent,
    EngineEventType,
    PIECE_POINTS
)

print("Nạp module packages/commander_game_engine/engine thành công!")

Nạp module packages/commander_game_engine/engine thành công!


## 1. Tính bất biến của trạng thái đầu vào (State Immutability)

In [2]:
state_orig = create_initial_state()
tank_pos_before = state_orig.pieces["red_tank_1"].position
player_before = state_orig.current_player

# Thực hiện 1 nước đi hợp lệ
act = AtomicAction(ActionType.MOVE, "red_tank_1", tank_pos_before, (tank_pos_before[0] + 1, tank_pos_before[1]))
result = apply_action(state_orig, act)

assert result.accepted == True
assert result.state is not state_orig, "Engine phải tạo một đối tượng GameState mới!"

# Kiểm tra trạng thái gốc TUYỆT ĐỐI KHÔNG BỊ SỬA ĐỔI
assert state_orig.pieces["red_tank_1"].position == tank_pos_before
assert state_orig.current_player == player_before
assert state_orig.turn_number == 1

# Kiểm tra trạng thái mới
assert result.state.pieces["red_tank_1"].position == (tank_pos_before[0] + 1, tank_pos_before[1])
assert result.state.current_player == Player.BLUE

print("PASS: Trạng thái nguồn hoàn toàn bất biến (Immutable) sau khi áp dụng nước đi!")

PASS: Trạng thái nguồn hoàn toàn bất biến (Immutable) sau khi áp dụng nước đi!


## 2. Thực thi nước đi thường (`MOVE`) & Sự kiện phát sinh

In [3]:
state = create_initial_state()
inf_pos = state.pieces["red_infantry_1"].position
act_inf = AtomicAction(ActionType.MOVE, "red_infantry_1", inf_pos, (inf_pos[0] + 1, inf_pos[1]))
res = apply_action(state, act_inf)

assert res.accepted == True
assert res.state.get_piece_at(inf_pos) is None, f"Ô xuất phát {inf_pos} phải được giải phóng!"
assert res.state.get_piece_at((inf_pos[0] + 1, inf_pos[1])).id == "red_infantry_1", "Ô đích phải có quân Bộ binh!"

# Kiểm tra sự kiện
event_types = [e.event_type for e in res.events]
assert EngineEventType.MOVED in event_types
assert EngineEventType.TURN_CHANGED in event_types
print(f"PASS: Nước đi MOVE thành công, phát sinh các sự kiện: {[e.event_type.value for e in res.events]}")

PASS: Nước đi MOVE thành công, phát sinh các sự kiện: ['moved', 'turn_changed']


## 3. Thực thi nước ăn quân thế chỗ (`CAPTURE`) & Bảng điểm

In [4]:
# Bảng điểm chuẩn theo đặc tả
df_points = pd.DataFrame(list(PIECE_POINTS.items()), columns=["Loại quân", "Điểm số"]).sort_values(by="Điểm số", ascending=False)
display(df_points)

# Tạo thế cờ ăn quân
st_cap = GameState()
tank_red = Piece(id="red_tank", piece_type=PieceType.TANK, owner=Player.RED, position=(4, 4))
inf_blue = Piece(id="blue_inf", piece_type=PieceType.INFANTRY, owner=Player.BLUE, position=(4, 5))
st_cap.add_piece(tank_red)
st_cap.add_piece(inf_blue)
st_cap.current_player = Player.RED

act_cap = AtomicAction(ActionType.CAPTURE, "red_tank", (4, 4), (4, 5), target_piece_id="blue_inf")
res_cap = apply_action(st_cap, act_cap)

assert res_cap.accepted == True
assert "blue_inf" not in res_cap.state.pieces, "Quân địch bị bắt phải bị loại khỏi bàn cờ!"
assert res_cap.state.get_piece_at((4, 5)).id == "red_tank", "Xe tăng phải thế chỗ tại (4, 5)!"

# Kiểm tra sự kiện cộng điểm
cap_events = [e for e in res_cap.events if e.event_type == EngineEventType.POINTS_AWARDED]
assert len(cap_events) == 1
assert cap_events[0].points == PIECE_POINTS[PieceType.INFANTRY]
print(f"PASS: Ăn quân thành công! Đã cộng {cap_events[0].points} điểm (chuẩn điểm Bộ binh = {PIECE_POINTS[PieceType.INFANTRY]}).")

                 Loại quân  Điểm số
0      PieceType.COMMANDER      100
8           PieceType.NAVY       80
7       PieceType.AIRCRAFT       40
4      PieceType.ARTILLERY       30
6        PieceType.MISSILE       20
3           PieceType.TANK       20
1       PieceType.INFANTRY       10
2        PieceType.MILITIA       10
5  PieceType.ANTI_AIRCRAFT       10
9       PieceType.ENGINEER       10
PASS: Ăn quân thành công! Đã cộng 10 điểm (chuẩn điểm Bộ binh = 10).


## 4. Thực thi bắn hỏa lực tại chỗ (`BOMBARD`)

In [5]:
# Pháo đất liền bắn Tàu trên biển
st_bomb = GameState()
art_red = Piece(id="red_art", piece_type=PieceType.ARTILLERY, owner=Player.RED, position=(3, 2))
navy_blue = Piece(id="blue_navy", piece_type=PieceType.NAVY, owner=Player.BLUE, position=(3, 0))
st_bomb.add_piece(art_red)
st_bomb.add_piece(navy_blue)
st_bomb.current_player = Player.RED

act_bomb = AtomicAction(ActionType.BOMBARD, "red_art", (3, 2), (3, 0), target_piece_id="blue_navy")
res_bomb = apply_action(st_bomb, act_bomb)

assert res_bomb.accepted == True
assert "blue_navy" not in res_bomb.state.pieces, "Tàu chiến địch phải bị tiêu diệt!"
assert res_bomb.state.get_piece_at((3, 2)).id == "red_art", "Pháo binh PHẢI GIỮ NGUYÊN VỊ TRÍ ĐẤT LIỀN (3, 2)!"
assert res_bomb.state.get_piece_at((3, 0)) is None, "Ô biển (3, 0) phải trống, Pháo không được thế chỗ lên biển!"

pts = sum(e.points for e in res_bomb.events if e.event_type == EngineEventType.POINTS_AWARDED)
assert pts == PIECE_POINTS[PieceType.NAVY]
print(f"PASS: BOMBARD bắn hạ tàu chiến thành công! Pháo giữ nguyên tọa độ, cộng +{pts} điểm Hải quân.")

PASS: BOMBARD bắn hạ tàu chiến thành công! Pháo giữ nguyên tọa độ, cộng +80 điểm Hải quân.


## 5. Điều kiện kết thúc ván cờ (Terminal Victory)

In [6]:
# Thế cờ: Xe tăng Đỏ bắt Tư lệnh Xanh
st_win = GameState()
tank_red = Piece(id="red_tank", piece_type=PieceType.TANK, owner=Player.RED, position=(5, 5))
cmd_blue = Piece(id="blue_cmd", piece_type=PieceType.COMMANDER, owner=Player.BLUE, position=(5, 6))
st_win.add_piece(tank_red)
st_win.add_piece(cmd_blue)
st_win.current_player = Player.RED

act_kill_cmd = AtomicAction(ActionType.CAPTURE, "red_tank", (5, 5), (5, 6), target_piece_id="blue_cmd")
res_win = apply_action(st_win, act_kill_cmd)

assert res_win.accepted == True
assert res_win.terminal == True, "Ván đấu phải chuyển sang trạng thái kết thúc (terminal=True)!"
assert res_win.winner == Player.RED, "Phe Đỏ phải được công nhận chiến thắng!"

event_types = [e.event_type for e in res_win.events]
assert EngineEventType.GAME_WON in event_types
print(f"PASS: Tiêu diệt Tư lệnh đối phương -> Trận đấu kết thúc ngay lập tức! Người thắng: {res_win.winner.value.upper()}.")

PASS: Tiêu diệt Tư lệnh đối phương -> Trận đấu kết thúc ngay lập tức! Người thắng: RED.


## 6. Xử lý từ chối hành động không hợp lệ

In [7]:
state = create_initial_state()

# Thử đi quân sai phe (Phe Xanh đi khi đang lượt Đỏ)
act_wrong_player = AtomicAction(ActionType.MOVE, "blue_tank_1", (8, 3), (7, 3))
res_rej = apply_action(state, act_wrong_player)

assert res_rej.accepted == False
assert res_rej.verdict.valid == False
assert res_rej.state.current_player == Player.RED, "Trạng thái không được chuyển lượt khi bị từ chối!"
print(f"PASS: Hành động sai lượt bị từ chối an toàn: {res_rej.verdict.reason}")

PASS: Hành động sai lượt bị từ chối an toàn: Chưa đến lượt của quân này.


## 7. Kết luận Engine
Module `engine.py` hoàn thành xuất sắc toàn bộ trách nhiệm chuyển trạng thái (State Transition):
- Bảo vệ tuyệt đối tính bất biến của trạng thái gốc.
- Thực thi chuẩn xác các hành động `MOVE`, `CAPTURE`, `BOMBARD`.
- Tính điểm chính xác theo bảng giá trị quân cờ `PIECE_POINTS`.
- Phát hiện điều kiện kết thúc trận đấu và công nhận người thắng cuộc.